# 🚢 Maritime Route Optimizer — Kepler.gl Visualization

Interactive visualization of:
- Real AIS vessel trajectories
- Port graph (nodes + edges)
- Optimized routes with GNN-predicted costs

**Output:** `app/maritime_routes.html` — standalone interactive map

## 0. Imports

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from keplergl import KeplerGl
import sys
sys.path.append('..')

print('✅ Imports OK')

## 1. Load Data

In [ ]:
# Load clean AIS trajectories (sample for performance)
ais = pd.read_parquet('../data/processed/ais_features.parquet')

# Sample 50k points for smooth rendering
ais_sample = ais.sample(50_000, random_state=42).copy()
ais_sample['timestamp'] = ais_sample['timestamp'].astype(str)

print(f'AIS sample: {len(ais_sample):,} points')
print(f'Columns: {list(ais_sample.columns)}')

In [ ]:
# Load port graph
nodes = pd.read_parquet('../data/processed/graph_nodes.parquet')
edges = pd.read_parquet('../data/processed/graph_edges.parquet')

print(f'Ports (nodes): {len(nodes)}')
print(f'Routes (edges): {len(edges)}')
nodes.head()

## 2. Prepare Layers

In [ ]:
# Layer 1: AIS trajectories
trajectories = ais_sample[['mmsi', 'lat', 'lon', 'sog', 'vessel_category', 'timestamp']].copy()
trajectories = trajectories.rename(columns={'sog': 'speed_knots'})
print('Layer 1 - Trajectories:', trajectories.shape)

In [ ]:
# Layer 2: Port nodes
ports_layer = nodes[['name', 'country', 'lat', 'lon']].copy()
print('Layer 2 - Ports:', ports_layer.shape)

In [ ]:
# Layer 3: Route edges (arc layer)
edges_layer = edges.merge(
    nodes[['port_id', 'lat', 'lon', 'name']].rename(
        columns={'port_id': 'departure_port_id', 'lat': 'src_lat', 'lon': 'src_lon', 'name': 'src_name'}
    ),
    on='departure_port_id', how='left'
).merge(
    nodes[['port_id', 'lat', 'lon', 'name']].rename(
        columns={'port_id': 'arrival_port_id', 'lat': 'dst_lat', 'lon': 'dst_lon', 'name': 'dst_name'}
    ),
    on='arrival_port_id', how='left'
)
edges_layer = edges_layer[[
    'src_name', 'dst_name', 'src_lat', 'src_lon',
    'dst_lat', 'dst_lon', 'n_vessels', 'avg_speed', 'avg_distance_km'
]].dropna()
print('Layer 3 - Edges:', edges_layer.shape)

In [ ]:
# Layer 4: Optimized routes via GNN + A*
from src.models.optimizer import MaritimeRouteOptimizer

opt = MaritimeRouteOptimizer(
    nodes_path='../data/processed/graph_nodes.parquet',
    edges_path='../data/processed/graph_edges.parquet',
    model_path='../data/processed/gnn_model.pt',
)

route_pairs = [
    ('Los Angeles', 'Long Beach'),
    ('New Orleans', 'Gretna'),
    ('Norfolk', 'Newport News'),
    ('Houston', 'Baytown'),
    ('San Francisco', 'Oakland'),
]

optimized_routes = []
for origin, dest in route_pairs:
    result = opt.optimize(origin, dest)
    if result.found:
        for i, (lat, lon) in enumerate(result.path_coords):
            optimized_routes.append({
                'route': f'{result.origin} → {result.destination}',
                'step': i,
                'lat': lat,
                'lon': lon,
                'port_name': result.path_ports[i],
                'total_cost': result.total_cost,
                'total_distance_km': result.total_distance_km,
            })

routes_layer = pd.DataFrame(optimized_routes)
print(f'Layer 4 - Optimized routes: {len(routes_layer)} points, {routes_layer["route"].nunique()} routes')

## 3. Build & Display Map

In [ ]:
config = {
    'version': 'v1',
    'config': {
        'mapState': {
            'latitude': 32.0, 'longitude': -95.0,
            'zoom': 4, 'bearing': 0, 'pitch': 0,
        },
        'mapStyle': {'styleType': 'dark'},
    }
}

map_ = KeplerGl(height=700, config=config)
map_.add_data(data=trajectories, name='AIS Trajectories')
map_.add_data(data=ports_layer, name='Port Nodes')
map_.add_data(data=edges_layer, name='Route Edges')
map_.add_data(data=routes_layer, name='Optimized Routes')

map_

## 4. Export to HTML

In [ ]:
output_path = '../app/maritime_routes.html'
Path('../app').mkdir(exist_ok=True)
map_.save_to_html(file_name=output_path)
print(f'✅ Map saved to {output_path}')
print(f'   Open: file://{Path(output_path).resolve()}')